# Week 3 Tools Experiment

Tool loops, tool descriptions, and live web search with Gemini.

In [1]:
import os
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent

load_dotenv(project_root / '.env')
expected_python = (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert Path(sys.executable).resolve() == expected_python
assert sys.version_info[:3] == (3, 13, 1)
model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')
print('Using the project .venv, Python 3.13.1, and Gemini Flash-Lite.')

Using the project .venv, Python 3.13.1, and Gemini Flash-Lite.


## Experiment 1: Build a tool and inspect the loop

In [2]:
def _metric_values(true_positives: int, false_positives: int, false_negatives: int) -> dict[str, float]:
    precision = true_positives / (true_positives + false_positives) if true_positives + false_positives else 0.0
    recall = true_positives / (true_positives + false_negatives) if true_positives + false_negatives else 0.0
    f1_score = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score}

@tool(
    'calculate_detection_metrics',
    description=(
        'Calculate binary detection precision, recall, and F1 score from counts. '
        'Provide true_positives, false_positives, and false_negatives as non-negative integer counts. '
        'Precision is TP / (TP + FP), recall is TP / (TP + FN), and F1 is the harmonic mean of precision and recall. '
        'Returns a dictionary containing precision, recall, and f1_score.'
    ),
)
def calculate_detection_metrics(true_positives: int, false_positives: int, false_negatives: int) -> dict[str, float]:
    return _metric_values(true_positives, false_positives, false_negatives)

metrics_question = 'Use the available tool to calculate the detection metrics for 42 true positives, 6 false positives and 9 false negatives.'
metrics_agent = create_agent(model=model, tools=[calculate_detection_metrics])
metrics_result = metrics_agent.invoke({'messages': [HumanMessage(content=metrics_question)]})

for message in metrics_result['messages']:
    if isinstance(message, HumanMessage):
        role = 'HUMAN MESSAGE'
    elif isinstance(message, AIMessage) and message.tool_calls:
        role = 'AI MESSAGE CONTAINING TOOL CALL'
    elif isinstance(message, ToolMessage):
        role = 'TOOL RESPONSE'
    elif isinstance(message, AIMessage):
        role = 'FINAL AI ANSWER'
    else:
        role = 'OTHER MESSAGE'
    print(f'\n--- {role} ---')
    pprint(message)

tool_call_message = next(message for message in metrics_result['messages'] if isinstance(message, AIMessage) and message.tool_calls)
tool_call_text = tool_call_message.content
text_is_minimal = not tool_call_text.strip() if isinstance(tool_call_text, str) else not tool_call_text
print('AI tool-call message has empty or minimal text content:', text_is_minimal)
print('Requested call is stored in tool_calls:', bool(tool_call_message.tool_calls))


--- HUMAN MESSAGE ---
HumanMessage(content='Use the available tool to calculate the detection metrics for 42 true positives, 6 false positives and 9 false negatives.', additional_kwargs={}, response_metadata={}, id='3e54998b-6db4-4544-b80c-f48394238e71')

--- AI MESSAGE CONTAINING TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'calculate_detection_metrics', 'arguments': '{"true_positives": 42, "false_positives": 6, "false_negatives": 9}'}, '__gemini_function_call_thought_signatures__': {'call_361861': 'EnMKcQFpFH0T+HNlSXjSJGtVxtzPCc082TAUcyz/8jGcG+HBez8RwWIFo/ECyRz3qUtGEhPf6XoEXV9hQl/6SvH7/OTFXpHdON3rwefFPAxZwo/Su2XBfKjrYtHfPOFXT6iWEPpru1ga73564RBH6vG+/fZX'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd81-0e3f-7d93-8f12-2e29a70d1c51-0', tool_calls=[{'name': 'calculate_detection_metrics', 'args': {'true_positives': 42, 'false_positives': 6

## Experiment 2: Bad and good tool descriptions

In [3]:
@tool('utility', description='tool')
def vague_metrics_tool(true_positives: int, false_positives: int, false_negatives: int) -> dict[str, float]:
    return _metric_values(true_positives, false_positives, false_negatives)

vague_agent = create_agent(model=model, tools=[vague_metrics_tool])
vague_result = vague_agent.invoke({'messages': [HumanMessage(content=metrics_question)]})
vague_calls = [call for message in vague_result['messages'] if isinstance(message, AIMessage) for call in message.tool_calls]
print('Bad description tool calls:')
pprint(vague_calls)
vague_selected_tool = any(call['name'] == 'utility' for call in vague_calls)
print('Selected the vague tool:', vague_selected_tool)

Bad description tool calls:
[{'args': {'false_negatives': 9, 'false_positives': 6, 'true_positives': 42},
  'id': 'call_293224',
  'name': 'utility',
  'type': 'tool_call'}]
Selected the vague tool: True


In [4]:
@tool(
    'calculate_detection_metrics',
    description=(
        'Calculate binary detection precision, recall, and F1 score from true_positives, false_positives, and false_negatives. '
        'Precision = TP / (TP + FP); recall = TP / (TP + FN); F1 is the harmonic mean of precision and recall. '
        'Use integer counts and return all three values.'
    ),
)
def clearly_described_metrics_tool(true_positives: int, false_positives: int, false_negatives: int) -> dict[str, float]:
    return _metric_values(true_positives, false_positives, false_negatives)

clear_agent = create_agent(model=model, tools=[clearly_described_metrics_tool])
clear_result = clear_agent.invoke({'messages': [HumanMessage(content=metrics_question)]})
clear_calls = [call for message in clear_result['messages'] if isinstance(message, AIMessage) for call in message.tool_calls]
print('Good description tool calls:')
pprint(clear_calls)
clear_selected_tool = any(call['name'] == 'calculate_detection_metrics' for call in clear_calls)
print('Selected the clearly described tool:', clear_selected_tool)
print('Tool-call comparison:')
pprint({'vague_tool_calls': vague_calls, 'clear_tool_calls': clear_calls})

Good description tool calls:
[{'args': {'false_negatives': 9, 'false_positives': 6, 'true_positives': 42},
  'id': 'call_232985',
  'name': 'calculate_detection_metrics',
  'type': 'tool_call'}]
Selected the clearly described tool: True
Tool-call comparison:
{'clear_tool_calls': [{'args': {'false_negatives': 9,
                                'false_positives': 6,
                                'true_positives': 42},
                       'id': 'call_232985',
                       'name': 'calculate_detection_metrics',
                       'type': 'tool_call'}],
 'vague_tool_calls': [{'args': {'false_negatives': 9,
                                'false_positives': 6,
                                'true_positives': 42},
                       'id': 'call_293224',
                       'name': 'utility',
                       'type': 'tool_call'}]}


## Experiment 3: Find the model’s knowledge edge

This response has no live search evidence and should not be treated as a verified current rate.

In [5]:
eur_question = 'What is today’s EUR to Algerian dinar exchange rate? Give the date and cite your source.'
no_search_agent = create_agent(model=model)
no_search_result = no_search_agent.invoke({'messages': [HumanMessage(content=eur_question)]})
no_search_response = no_search_result['messages'][-1]
print('Gemini response without live search (unverified):')
pprint(no_search_response)

Gemini response without live search (unverified):
AIMessage(content=[{'type': 'text', 'text': "As of **May 22, 2024**, the official exchange rate for 1 Euro (EUR) to Algerian Dinar (DZD) is approximately **145.45 DZD**.\n\n**Source:** [Bank of Algeria (Banque d'Algérie)](https://www.bank-of-algeria.dz/)\n\n***\n\n**Important Note:** The official rate published by the Bank of Algeria is used for banking and official transactions. If you are looking for the rate on the **parallel market (informal market)**, it is significantly higher, often fluctuating between 240 and 245 DZD per Euro, depending on the specific location and current demand.", 'extras': {'signature': 'EnMKcQFpFH0T2WW7AbtaBay0DcwaufzBMRVeBAckHGA9v3gD2vcoxL1mK5oZLxRy7k/T6VsvYTpF3XrCOSffNBEfxjXcAZv4oiG83e87pfAzyg9n+svawNLsvdvxRfMQQjRbo0yE31yPyAu9lk1Qdz+bkES3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id=

## Experiment 4: Add web search

In [7]:
tavily_key_available = bool(os.getenv('TAVILY_API_KEY'))
print('TAVILY_API_KEY available:', tavily_key_available)

if not tavily_key_available:
    print('Search experiment skipped. Add TAVILY_API_KEY to the ignored local .env file, then rerun this cell.')
else:
    from typing import Any
    from tavily import TavilyClient

    tavily_client = TavilyClient()

    @tool('web_search', description='Search the web for current exchange-rate information and return result titles, URLs, and snippets.')
    def web_search(query: str) -> dict[str, Any]:
        return tavily_client.search(query)

    search_system_prompt = (
        'Use web search for current exchange rates. Quote each source date exactly as written; never change a year or infer a date. '
        'State the reported EUR-to-DZD rate and unit, and cite the source name and URL. Compare every returned result containing a numeric rate. '
        'If rates or dates differ, explicitly warn that sources disagree and list each reported value, date, and source. '
        'If results do not establish a current rate and date, say the rate cannot be verified. Do not invent missing details.'
    )
    search_agent = create_agent(model=model, tools=[web_search], system_prompt=search_system_prompt)
    search_result = search_agent.invoke({'messages': [HumanMessage(content=eur_question)]})

    for message in search_result['messages']:
        if isinstance(message, HumanMessage):
            role = 'HUMAN MESSAGE'
        elif isinstance(message, AIMessage) and message.tool_calls:
            role = 'AI MESSAGE CONTAINING SEARCH TOOL CALL'
        elif isinstance(message, ToolMessage):
            role = 'SEARCH TOOL RESPONSE'
        elif isinstance(message, AIMessage):
            role = 'FINAL AI ANSWER'
        else:
            role = 'OTHER MESSAGE'
        print(f'\n--- {role} ---')
        pprint(message)

TAVILY_API_KEY available: True

--- HUMAN MESSAGE ---
HumanMessage(content='What is today’s EUR to Algerian dinar exchange rate? Give the date and cite your source.', additional_kwargs={}, response_metadata={}, id='4f9d356c-9b60-4f53-abcc-8d05dc3bd765')

--- AI MESSAGE CONTAINING SEARCH TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "current EUR to DZD exchange rate"}'}, '__gemini_function_call_thought_signatures__': {'call_237154': 'EnMKcQFpFH0T7RV81+RaAvMlVeQ8Iq35FPqO9EkKZydDYPywRHRNV0YP9TZod0jYvBK8E8xIg5XzPHvL2v295Ka6qon6rrew2bnNczz79I0y1q5y4fFftcHlHyG1/ZqYgDjjGqYENkaiaSckKfxFfEE8z9Gc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fd82-e440-71f2-b487-5be2b8647ca5-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current EUR to DZD exchange rate'}, 'id': 'call_237154', 'type': 'tool_call'}], in

## Observations

- **What the tool-call loop looked like**  
  The sequence contained the original human message, an AI message with empty text but a populated `tool_calls` field, the tool response and the final AI answer. The calculated values were precision 0.875, recall 0.8235 and F1 0.8485.

- **What changed after improving the tool description**  
  Both the vague and clearly described tools were selected in this simple experiment because each agent had only one available tool. The clearer description still made the purpose, inputs and expected result easier to understand. This would become more important when the agent must choose between several tools.

- **What the model said without live search**  
  Without a search tool, the model confidently reported 145.45 DZD/EUR using a date from 2024, even though the question asked for today’s rate. It also included an unsupported parallel-market estimate. The answer sounded credible but did not contain current evidence.

- **What changed after adding live search**  
  The agent called Tavily and retrieved several sources with different rates. This exposed disagreement between official, platform and parallel-market figures. However, the model still associated some dates with sources that did not clearly establish those dates.

- **Why tool results should still be checked**  
  Tools give the model access to external information, but the model can still misinterpret dates, combine incompatible figures or describe uncertain evidence too confidently. The tool call, raw result, source date and final answer all need to be reviewed separately.